# (노트) 텐서플로우와 미분

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [데이터과학]

In [1]:
import tensorflow as tf
import numpy as np
import tensorflow.experimental.numpy as tnp
#tnp.experimental_enable_numpy_behavior()

In [2]:
tf.config.experimental.list_physical_devices('GPU')

2022-03-21 07:07:39.872345: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:939] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

### tf.Variable

#### 선언 

`-` tf.Variable()로 선언

In [3]:
tf.Variable([1,2,3,4])

<tf.Variable 'Variable:0' shape=(4,) dtype=int32, numpy=array([1, 2, 3, 4], dtype=int32)>

`-` tf.constant() 선언후 변환 

In [4]:
a=tf.constant([1,2,3,4])
a=tf.Variable(a) 
a

<tf.Variable 'Variable:0' shape=(4,) dtype=int32, numpy=array([1, 2, 3, 4], dtype=int32)>

#### 타입

In [5]:
type(tf.Variable(1))

tensorflow.python.ops.resource_variable_ops.ResourceVariable

#### 인덱싱 

In [6]:
a=tf.Variable([1,2,3,4])
a[:2]

<tf.Tensor: shape=(2,), dtype=int32, numpy=array([1, 2], dtype=int32)>

#### tf.Variable $\to$ 넘파이 

In [7]:
tf.Variable([1,2,3,4]).numpy()

array([1, 2, 3, 4], dtype=int32)

#### tf.Variable 도 불편하다. 

In [8]:
tf.Variable([1,2,3])+tf.Variable([1.0,2.0,3.0]) # 이거 원래 실행못함. 그런데 여기서 실행되는 이유는 tnp덕임

InvalidArgumentError: cannot compute AddV2 as input #1(zero-based) was expected to be a int32 tensor but is a float tensor [Op:AddV2]

#### 연산 

In [9]:
tf.Variable([1,2,3])+tf.Variable([1,2,3]) # 연산가능

<tf.Tensor: shape=(3,), dtype=int32, numpy=array([2, 4, 6], dtype=int32)>

In [10]:
_tmp = tf.Variable([1,2,3])+tf.Variable([1,2,3]) 
type(_tmp) #그런데 연산결과는 tf.constant 이다? 

tensorflow.python.framework.ops.EagerTensor

#### 형태변환 

In [11]:
_tmp = tf.reshape(tf.Variable([1,2,3,4]),[2,2])
_tmp

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[1, 2],
       [3, 4]], dtype=int32)>

In [12]:
type(_tmp) # 또 연산결과는 tf.constant가 되었음 

tensorflow.python.framework.ops.EagerTensor

#### 선언고급 

In [13]:
a=tf.Variable(np.diag([1,2,3]))
a

<tf.Variable 'Variable:0' shape=(3, 3) dtype=int64, numpy=
array([[1, 0, 0],
       [0, 2, 0],
       [0, 0, 3]])>

#### tf.concat 

In [14]:
a=tf.Variable([1,2])
b=tf.Variable([3,4])
c=tf.concat([a,b],axis=0)
c

<tf.Tensor: shape=(4,), dtype=int32, numpy=array([1, 2, 3, 4], dtype=int32)>

#### tf.stack

In [15]:
a=tf.Variable([1,2])
b=tf.Variable([3,4])
c=tf.stack([a,b],axis=0)
c

<tf.Tensor: shape=(2, 2), dtype=int32, numpy=
array([[1, 2],
       [3, 4]], dtype=int32)>

In [16]:
a=tf.Variable([1,2,3,4])

#### 심지어 tf.Variable()로 만들어진 오브젝트는 tnp의 효과(은총)도 받지 못함

In [17]:
tf.Variable([1,2,3,4]).reshape(2,2)

AttributeError: 'ResourceVariable' object has no attribute 'reshape'

#### 변수값변경가능(?)

`-` Variable과 constant선언 

In [269]:
a=tf.Variable([1,2])
b=tf.constant([1,2])

`-` `a.`+ `tab`와 `b.`+ `tab`은 출력가능한 목록부터 다르다. a에는 assing_add라는 메소드가 있는데 이것을 이용하여 a의 값을 바꿀 수 있다. 

In [272]:
a.assign_add([2,3])

<tf.Variable 'UnreadVariable' shape=(2,) dtype=int32, numpy=array([5, 8], dtype=int32)>

In [271]:
a

<tf.Variable 'Variable:0' shape=(2,) dtype=int32, numpy=array([3, 5], dtype=int32)>

#### 요약 

`-` tf.Variable()로 만들어야 하는 뚜렷한 차이를 모르겠음. 

`-` 애써 tf.Variable()로 만들어도 간단한연산을 하면 그 결과는 tf.constant()로 만든 오브젝트와 동일해짐. 

### 미분 

`-` 예제: 컴퓨터를 이용하여 $x=2$에서 $y=3x^2$의 접선의 기울기를 구해보자. 

(손풀이) 

$\frac{dy}{dx}=6x$ 이므로, $x=2$를 대입하면 답은 12이다. 

(컴퓨터로풀이) 

도함수를 어떻게 구하지? 

통찰: 컴퓨터에게 도함수를 구하라고 하는것은 어려운데 특정점에서 도함수값을 구하는것은 그렇게 어렵지 않다. 

In [297]:
x1=2
y1=3*x1**2 

In [300]:
x2=2+0.0001
y2=3*x2**2 

In [301]:
(y2-y1)/(x2-x1)

12.000300000007059

위의 과정을 아예 함수화 시켜서 임의의 점에서 도함수값을 출력하게 할 수도 있다. 

In [302]:
def d(f,x):
    return (f(x+0.0001) - f(x))/0.0001

In [308]:
def f(x):
    return 3*x**2 

In [310]:
d(f,2) 

12.000300000032382

lambda에 익숙하다면 아래도 가능하다는것을 알것이다. 

In [302]:
def d(f,x):
    return (f(x+0.0001) - f(x))/0.0001

In [311]:
d(lambda x: 3*x**2,2) 

12.000300000032382

`-` tf를 이용한 미분: 나도 미분할수는 있는데 점점 복잡한 미분을 계산하려면 좀 더 좋은 도구가 필요하다. 

In [312]:
def f(x,y):
    return 3*x**2 + 2*y 

In [313]:
d(f,(3,2))

TypeError: can only concatenate tuple (not "float") to tuple

내가 만든건 아쉽게도 그렇게 확장성이 좋지 못하다. (2차원을 또 만들어야 하나..?)

`-` 예제1

In [314]:
x = tf.Variable(2.0)
a = tf.constant(3.0) 

In [315]:
mytape=tf.GradientTape() # 테이프 오브젝트 생성
mytape.__enter__() ## 테이프에 기록시작 
y = a*x**2 ## y=3x^2 이야!
mytape.__exit__(None,None,None) ## 테이프에 기록끝

In [316]:
mytape.gradient(y,x)

<tf.Tensor: shape=(), dtype=float32, numpy=12.0>

`-` 예제2

In [317]:
x = tf.Variable(2.0)
# a = tf.constant(3.0) 
mytape=tf.GradientTape() ## 테이프 오브젝트생성 
mytape.__enter__() ## 테이프에 기록시작 
a=x/2*3 # a=x/2*3 임 
y = a*x**2 ## y=ax^2=(3/2)x^3 으로 인식한다! 
mytape.__exit__(None,None,None) ## 테이프에 기록끝 
mytape.gradient(y,x) # 미분해봐

<tf.Tensor: shape=(), dtype=float32, numpy=18.0>

(손풀이) 

$y=\frac{3}{2}x^3$ 

$\frac{d}{dx}y = \frac{3}{2}3x^2$ 

$x=2$를 넣으면 

In [289]:
1.5*3*4 

18.0

`-` 테이프의 개념 

(상황) 

우리가 어려운 미분계산을 컴퓨터에게 부탁하는 상황임. (예를들면 $y=3x^2$) 컴퓨터에게 부탁을 하기 위해서는 연습장(=테이프)에 $y=3x^2$이라는 수식을 써서 보여줘야하는데 이때 컴퓨터에게 target이 무엇인지 그리고 무엇으로 미분하고 싶은 것인지를 명시해야함. 

(1) `mytape = tf.GradientTape()`: 연습장 오브젝트를 하나 만듬, 만들어진 연습장에 mytape라는 제목을 씀. 

(2) `mytape.__enter__()`: mytape 연습장을 "열고" 기록할 준비를 함. 

(3) `a=x/2*3; y=a*x**2`: 연습장에 수식을 씀. 이때 컴퓨터는 $x$를 변수라고 인식 (왜? tf.Variable()로 선언되었으니까) 

(4) `mytape.__exit__(None,None,None)`: 기록을 끝내고 연습장을 "닫음". 

(5) `mytape.gradient(y,x)`: y를 x로 미분하라는 메모를 남기고 연습장을 컴퓨터에게 전달. 컴퓨터는 계산결과를 display. 

`-` 예제3

In [319]:
x = tf.Variable(2.0)
a=x/2*3 
mytape=tf.GradientTape()
mytape.__enter__() ## 기록시작 
y = a*x**2 ## 이번엔 y=3x^2 으로 인식
mytape.__exit__(None,None,None) ## 기록끝 
mytape.gradient(y,x) # 미분해봐

<tf.Tensor: shape=(), dtype=float32, numpy=12.0>

이것은 직전 예제와 비슷해보이는데, 연습장에 $a=\frac{3}{2}x$를 쓰지 않았으므로 $y=ax^2=\frac{3}{2}x^3$이라고 컴퓨터는 인식하지 못함 

애초에 x를 tf.Variable로 만들어도, 그것을 연산한 결과 a는 tf.constant로 되었다. 

In [320]:
type(x)

tensorflow.python.ops.resource_variable_ops.ResourceVariable

In [321]:
type(x/2*3)

tensorflow.python.framework.ops.EagerTensor

(이해) 언제 연습장을 열고 닫을지 결정하는 것은 중요함 

`-` 예제4

(불편한점) 

예제3에 의해서 연습장을 언제 열고 언제 닫는지에 대한 미묘한 차이로 인해 컴퓨터가 인식하는 수식이 달라지고 그래서 결과가 달라짐을 이해함 

하지만 연습장을 맨날 열고/닫는 과정을 반복해야하는 것이 번거롭게 느껴짐 

이 과정을 생략할순 없으니 일종의 매크로 구문을 만들어서 간소화시키자! $\to$ with 문 

(불편해결) 

In [332]:
x = tf.Variable(2.0)
a = tf.constant(3.0) 
with tf.GradientTape() as mytape:
    y = a*x**2 

In [333]:
dy_dx = mytape.gradient(y, x) # dy/dx  = 6x , x=2 

In [334]:
dy_dx 

<tf.Tensor: shape=(), dtype=float32, numpy=12.0>

(코드분석) 
```python
with ... as myobject: 
    ## start with block 
    blabla~
    yadiyadi~
    ## end with block 
```


(1) `...`을 수행한 결과 생성되는 오브젝트를 myobject라고 이름붙인다. 

(2) with문이 시작되면서 myobject의 숨겨진 메소드 `.__enter__()`를 실행 

(3) blabla, yadiyadi 실행 (with문의 내용실행)

(4) with문이 끝날때 myobject의 숨겨진 메소드 `.__exit__()`를 실행 

`-` 예제5

In [336]:
x = tf.Variable(2.0)
with tf.GradientTape() as mytape:
    a = (3/2)*x
    y = a*x**2 

In [337]:
dy_dx = mytape.gradient(y, x) # dy/dx  = 6x , x=2 

In [338]:
dy_dx 

<tf.Tensor: shape=(), dtype=float32, numpy=18.0>

`-` 예제6: persistent = True

(관찰1)

In [356]:
x = tf.Variable(2.0)
a = tf.constant(3.0) 
with tf.GradientTape() as tape:
    y = a*x**2 

In [358]:
dy_dx = tape.gradient(y, x) # 2번이상 실행시켜서 에러메시지를 발생시켜보자.
dy_dx

RuntimeError: A non-persistent GradientTape can only be used to compute one set of gradients (or jacobians)

(관찰2)

In [352]:
x = tf.Variable(2.0)
a = tf.constant(3.0) 
with tf.GradientTape(persistent=True) as tape:
    y = a*x**2 

In [354]:
dy_dx = tape.gradient(y, x) # 2번 이상 실행해도 계속 계산해준다. 
dy_dx

<tf.Tensor: shape=(), dtype=float32, numpy=12.0>

`-` 예제7: watch

(관찰1)

In [375]:
x = tf.constant(2.0)
with tf.GradientTape(persistent=True) as tape:
    a = x*3/2
    y = a*x**2 

In [376]:
dy_dx = tape.gradient(y, x) # 뭘로 미분하라는거야 
print(dy_dx)

None


(관찰2)

In [377]:
x = tf.constant(2.0)
with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    a = x*3/2
    y = a*x**2 ## y=3/2 * x^3

In [379]:
dy_dx = tape.gradient(y,x) # 뭘로 미분하라는거야 
print(dy_dx)

tf.Tensor(18.0, shape=(), dtype=float32)


(깨달음) tf.Variable()로 선언된 타입은 tape.watch()를 사용하여 명시하지 않아도 알아서 감시대상이구나?
- 자동감시: tf.Variable()이용 
- 수동감시: tape.watch()이용

`-` 예제8 

In [389]:
x = tf.Variable(2.0)
with tf.GradientTape(persistent=True, watch_accessed_variables=False) as tape:
    a = x*3/2
    y = a*x**2 

In [390]:
dy_dx = tape.gradient(y,x) 
print(dy_dx)

None


- tf.Variable()로 선언되었더라도 'watch_accessed_variables=False' 이면 tf.constant()로 선언된 변수처럼 취급
- 'watch_accessed_variables=False'는 자동감시모드를 해제하라는 의미

위의 예제에서 x를 watch하면 x를 다시 변수처럼 본다. 

In [399]:
x = tf.Variable(2.0)
with tf.GradientTape(persistent=True, watch_accessed_variables=False) as tape: # 자동감시모드 해제 
    tape.watch(x) # 수동감시 
    a = x*3/2
    y = a*x**2 

In [551]:
dy_da = tape.gradient(y,x) 
print(dy_da)

tf.Tensor(18.0, shape=(), dtype=float32)


그냥 명시적으로 .watch(x)를 쓰는게 편하여 쓰는사람도 있다. (저에요!) 

In [556]:
x = tf.Variable(2.0)
with tf.GradientTape(persistent=True) as tape: 
    tape.watch(x) # 이미 자동감시 대상이니까 사실 수동감시명령을 내릴 필요는 없음, 즉 이건 필요없는 코드임
    a = x*3/2
    y = a*x**2 

In [557]:
tape.gradient(y,x) 

<tf.Tensor: shape=(), dtype=float32, numpy=18.0>

`-` 예제9

In [506]:
tnp.experimental_enable_numpy_behavior() # 이게 편하다고 했죠?

In [507]:
x = tnp.array([20.1, 22.2, 22.7, 23.3, 24.4, 25.1, 26.2, 27.3, 28.4, 30.4])
x

<tf.Tensor: shape=(10,), dtype=float64, numpy=array([20.1, 22.2, 22.7, 23.3, 24.4, 25.1, 26.2, 27.3, 28.4, 30.4])>

In [508]:
tnp.random.seed(43052)
y= x*2.2 + 10.2 + tnp.random.randn(10)
y

<tf.Tensor: shape=(10,), dtype=float64, numpy=
array([54.98269924, 60.27348365, 61.27621687, 60.53495888, 62.9770905 ,
       66.32168996, 66.87781372, 71.0050025 , 72.63837337, 77.11143943])>

$\beta_0 = 9$, $\beta_1=2$에서의 미분계수를 구해보자. 

(풀이) 

In [509]:
beta0= tf.Variable(9.0)
beta1= tf.Variable(2.0) 

In [510]:
with tf.GradientTape(persistent=True) as tape: 
    yhat = beta0 + beta1*x 
    loss = sum((y-yhat)**2)

In [512]:
tape.gradient(loss,beta0), tape.gradient(loss,beta1)

(<tf.Tensor: shape=(), dtype=float32, numpy=-127.597534>,
 <tf.Tensor: shape=(), dtype=float32, numpy=-3214.2532>)

`-` 예제10 

In [513]:
X = tnp.array([1.0]*10+[20.1, 22.2, 22.7, 23.3, 24.4, 25.1, 26.2, 27.3, 28.4, 30.4]).reshape(2,10).T
X

<tf.Tensor: shape=(10, 2), dtype=float64, numpy=
array([[ 1. , 20.1],
       [ 1. , 22.2],
       [ 1. , 22.7],
       [ 1. , 23.3],
       [ 1. , 24.4],
       [ 1. , 25.1],
       [ 1. , 26.2],
       [ 1. , 27.3],
       [ 1. , 28.4],
       [ 1. , 30.4]])>

In [536]:
betatrue = tnp.array([10.2,2.2]).reshape(2,1)
betatrue

<tf.Tensor: shape=(2, 1), dtype=float64, numpy=
array([[10.2],
       [ 2.2]])>

In [537]:
tnp.random.seed(43052)
y= X@betatrue + tnp.random.randn(10).reshape(10,1)
y

<tf.Tensor: shape=(10, 1), dtype=float64, numpy=
array([[54.98269924],
       [60.27348365],
       [61.27621687],
       [60.53495888],
       [62.9770905 ],
       [66.32168996],
       [66.87781372],
       [71.0050025 ],
       [72.63837337],
       [77.11143943]])>

(풀이) 

In [538]:
beta = tnp.array([9.0,2.0]).reshape(2,1)
beta

<tf.Tensor: shape=(2, 1), dtype=float64, numpy=
array([[9.],
       [2.]])>

In [539]:
with tf.GradientTape() as tape: 
    tape.watch(beta)
    yhat = X@beta 
    loss = (yhat-y).T @ (yhat-y) 

In [540]:
tape.gradient(loss,beta)

<tf.Tensor: shape=(2, 1), dtype=float64, numpy=
array([[ -127.59753624],
       [-3214.25306574]])>

(손풀이로 확인) 

loss의 도함수는 $\frac{\partial}{\partial \beta}loss = -2X'y+2X'X\beta$ 이므로

In [541]:
-2 * X.T @ y + 2* X.T @ X @ beta

<tf.Tensor: shape=(2, 1), dtype=float64, numpy=
array([[ -127.59753624],
       [-3214.25306574]])>

`-` 예제11: 위의 예제에서 이론적인 $\boldsymbol{\beta}$의 최적값을 찾아보고 (즉 $\hat{\boldsymbol{\beta}}$을 찾고) 그곳에서 loss의 미분을 구하라. 구한결과가 $\begin{bmatrix}0 \\ 0 \end{bmatrix}$ 임을 확인하라. 

(풀이) 

In [542]:
betahat = np.linalg.inv(X.T @ X) @ X.T @ y 
betahat

<tf.Tensor: shape=(2, 1), dtype=float64, numpy=
array([[12.10040012],
       [ 2.13112662]])>

In [543]:
with tf.GradientTape() as tape: 
    tape.watch(betahat) 
    yhat = X@betahat 
    loss = (y-yhat).T @ (y-yhat) 

In [544]:
tape.gradient(loss, betahat)

<tf.Tensor: shape=(2, 1), dtype=float64, numpy=
array([[4.83169060e-13],
       [1.18524917e-11]])>

(참고) $\beta$의 참값을 넣으면 저렇게 0으로 안나온다. 해보자.

In [558]:
betatrue

<tf.Tensor: shape=(2, 1), dtype=float64, numpy=
array([[10.2],
       [ 2.2]])>

In [546]:
with tf.GradientTape() as tape: 
    tape.watch(betatrue) 
    yhat = X@betatrue
    loss = (y-yhat).T @ (y-yhat) 

In [547]:
tape.gradient(loss, betatrue)

<tf.Tensor: shape=(2, 1), dtype=float64, numpy=
array([[ -3.55753624],
       [-76.87306574]])>

하지만 샘플의 수가 커지면 `tape.gradient(loss, betatrue)` $\approx$ `tape.gradient(loss, betahat)` 이 되겠지.
- 즉 betatrue $\approx$ betahat 이 된다는 의미 